# ReefScan-ID: Stage 1 segmentation

Stage 1 of the cascade: segment a diver photo into coral / substrate / other. Run in Google Colab with a T4 GPU runtime.

### Setup: Colab runtime and Google Drive

In [1]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU: {}".format(gpu_name))

base_dir = "/content/drive/MyDrive/ReefScan-ID"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))

ModuleNotFoundError: No module named 'torch'

### Setup: libraries

In [ ]:
!pip install -q datasets

import datasets
print("datasets version: {}".format(datasets.__version__))

## Section 1: Mask ID check

Question: which class ids appear in the Coralscapes masks, and do they match the 39 classes listed on the dataset card?

Checks: count the pixels of every mask value in the train and validation splits, then compare the values found with the ids on the card. The test split is left out here because it is kept for the final evaluation.

### Load the dataset

In [ ]:
raw_dir = base_dir + "/RawData"

import time
from datasets import load_dataset

print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
load_seconds = time.time() - start
print(ds)

cache_bytes = 0
for root, dirs, files in os.walk(raw_dir):
    for name in files:
        cache_bytes += os.path.getsize(os.path.join(root, name))
print("Cache folder size: {:.2f} GB".format(cache_bytes / 1024 ** 3))
print("Loading took {:.1f} seconds".format(load_seconds))

### Optional: copy the cache to the Colab disk

Set `use_local_copy = True` to copy the Drive cache to `/content` and load from there. Used to compare the loading time from Drive with the loading time from a local copy.

In [ ]:
import shutil

use_local_copy = False
local_dir = "/content/RawData"

if not use_local_copy:
    print("SKIPPED: local copy not requested")
else:
    start = time.time()
    if not os.path.isdir(local_dir):
        shutil.copytree(raw_dir, local_dir)
    print("Copy took {:.1f} seconds".format(time.time() - start))

    start = time.time()
    ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=local_dir)
    print("Loading from the local copy took {:.1f} seconds".format(time.time() - start))

### Class names from the dataset card

Ids and names copied from the table on the Coralscapes dataset card (read 2026-11-12).

In [ ]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

### Count pixels per mask value

In [ ]:
import numpy as np

pixel_counts = {}
for split in ["train", "validation"]:
    counts = np.zeros(256, dtype=np.int64)
    masks = ds[split].select_columns(["label"])
    for row in masks:
        mask = np.array(row["label"])
        counts += np.bincount(mask.ravel(), minlength=256)
    pixel_counts[split] = counts
    print("{}: {} masks counted, {} pixels".format(split, masks.num_rows, counts.sum()))

### Mask values found against the card

In [ ]:
import pandas as pd

train_counts = pixel_counts["train"]
val_counts = pixel_counts["validation"]
found = [int(v) for v in np.nonzero(train_counts + val_counts)[0]]

rows = []
for value in found:
    rows.append({
        "id": value,
        "name": class_names.get(value, "NOT ON THE CARD"),
        "train_pct": 100 * train_counts[value] / train_counts.sum(),
        "val_pct": 100 * val_counts[value] / val_counts.sum(),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False, float_format="{:.3f}".format))

not_on_card = [v for v in found if v not in class_names]
never_seen = [i for i in class_names if i not in found]
print("Mask values found: {}".format(len(found)))
print("Values found that are not on the card: {}".format(not_on_card))
print("Card ids that never appear in train or validation: {}".format(never_seen))

**Observation**: